# 🔍 Embeddings, Semantic Search and Vector Stores

### Tools and Techniques in Data Science — LangChain Module

| | |
|---|---|
| **Difficulty** | ⭐⭐ Intermediate → ⭐⭐⭐ Advanced |
| **Estimated Time** | 90–120 minutes |
| **Prerequisites** | Notebooks 01–03 |

---

**Welcome!** This notebook teaches you how to turn text into numbers (**embeddings**), store them efficiently (**vector stores**), and find semantically similar content (**similarity search**). These are the building blocks of RAG (Retrieval-Augmented Generation) — covered in the next notebook.

> 💡 **Data Science Focus:** You'll build a Data Science knowledge base and perform semantic search over it.

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Understand why keyword search is sometimes insufficient
2. Know what embeddings are and how they capture meaning
3. Compute semantic similarity between texts
4. Use embedding models (OpenAI API and local Ollama)
5. Create and query a vector store with ChromaDB
6. Build a Data Science knowledge base with metadata
7. Compare semantic search vs keyword search
8. Understand privacy, cost, and hardware tradeoffs

---

## ⚙️ Setup

In [ ]:
# Install packages if needed (uncomment)
# !pip install langchain langchain-openai langchain-ollama langchain-text-splitters langchain-chroma chromadb python-dotenv

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv

# LangChain core
from langchain_core.documents import Document

# Embeddings
from langchain_openai import OpenAIEmbeddings

# Text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store
from langchain_chroma import Chroma

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI API key:", "found" if api_key else "NOT SET")
print("All imports successful!")

---

## 1. Why Keyword Search Is Sometimes Insufficient

### The Problem

Traditional keyword search matches **exact words**. But humans don't always use the same words for the same concept.

| Query | Keyword Search Finds | Misses |
|---|---|---|
| "how to handle missing data" | Documents with "missing data" | "null values", "NaN", "empty fields" |
| "prediction model" | Documents with "prediction model" | "regression", "forecasting", "estimation" |
| "group similar items" | Documents with "group similar" | "clustering", "segmentation", "k-means" |

### Semantic Search Solves This

```mermaid
flowchart TD
    subgraph "Keyword Search"
        KQ["Query: 'missing values'"] --> KM["Match exact words"]
        KM --> KR["Finds: 'missing values' only"]
    end
    subgraph "Semantic Search"
        SQ["Query: 'missing values'"] --> SM["Understand meaning"]
        SM --> SR["Finds: 'NaN', 'null', 'empty fields', 'imputation'"]
    end
```

Semantic search understands **meaning**, not just words.

---

## 2. What Are Embeddings?

An **embedding** is a way to represent text as a **list of numbers** (a vector) that captures its **meaning**.

### Text → Vector

```mermaid
flowchart LR
    T["Text: 'Random Forest is an ensemble method'"] --> E["Embedding Model"]
    E --> V["Vector: [0.02, -0.15, 0.33, ..., 0.08]"]
```

### Key Properties

| Property | Description |
|---|---|
| **Fixed size** | All texts produce vectors of the same length (e.g., 1536 dimensions) |
| **Semantic** | Similar meanings → similar vectors |
| **Language-aware** | "car" and "automobile" have similar vectors |
| **Contextual** | "bank" (river) ≠ "bank" (financial) in good embeddings |

In [ ]:
# Create an embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed a single text
text = "Random Forest is an ensemble learning method."
vector = embeddings.embed_query(text)

print(f"Text: '{text}'")
print(f"Vector length: {len(vector)} dimensions")
print(f"First 5 values: {vector[:5]}")
print(f"Vector type: {type(vector).__name__}")

### 🔍 What Happened?

1. The embedding model converted text into a **1536-dimensional vector**
2. Each number represents some aspect of the text's meaning
3. Similar texts will have similar vectors

> 💡 **You don't need to understand what each number means.** What matters is that **similar texts produce similar vectors**.

---

## 3. Semantic Similarity

### Cosine Similarity

The most common way to measure similarity between two vectors is **cosine similarity** — the angle between them.

```
cosine_similarity = cos(angle between vectors)

1.0  = Identical meaning
0.0  = Completely unrelated
-1.0 = Opposite meaning (rare for text)
```

```mermaid
flowchart LR
    A["Similar texts\nSmall angle\nHigh similarity"] --> B["Different texts\nLarge angle\nLow similarity"]
```

In [ ]:
# Compute cosine similarity between texts
def cosine_similarity(vec_a, vec_b):
    """Calculate cosine similarity between two vectors."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Embed several data science texts
texts = [
    "Random Forest is an ensemble of decision trees.",
    "Decision trees split data based on feature values.",
    "K-Means groups data into clusters.",
    "Linear regression fits a straight line to data.",
    "An ensemble method combines multiple models."
]

vectors = embeddings.embed_documents(texts)

# Compare similarities
query = "What is an ensemble learning method?"
query_vector = embeddings.embed_query(query)

print(f"Query: '{query}'")
print("\nSimilarities:")
print("-" * 60)

for text, vec in zip(texts, vectors):
    sim = cosine_similarity(query_vector, vec)
    print(f"  {sim:.4f} | {text}")

### 🔍 What Happened?

The embedding model correctly identified that:
- **"Random Forest is an ensemble..."** and **"An ensemble method combines..."** are most similar to our query about ensemble learning
- **"K-Means groups data..."** and **"Linear regression..."** are less related

This is **semantic search** in action — matching by meaning, not keywords!

---

## 4. Embedding Models

### Chat Models vs Embedding Models

| | **Chat Model** | **Embedding Model** |
|---|---|---|
| **Purpose** | Generate text responses | Convert text to vectors |
| **Input** | Messages | Text strings |
| **Output** | Text (AIMessage) | List of floats (vector) |
| **Example** | GPT-4o, Llama 3.2 | text-embedding-3-small, nomic-embed-text |
| **Used for** | Conversations, generation | Search, similarity, clustering |

### API vs Local Embeddings

| | **OpenAI API** | **Local (Ollama)** |
|---|---|---|
| **Model** | text-embedding-3-small | nomic-embed-text, all-minilm |
| **Dimensions** | 1536 | Varies (384–768) |
| **Quality** | High | Good |
| **Cost** | Pay per token | Free |
| **Privacy** | Data sent to cloud | All data stays local |
| **Speed** | Fast (network) | Depends on hardware |

In [ ]:
# API-based embeddings (OpenAI)
api_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector = api_embeddings.embed_query("machine learning")
print(f"OpenAI embedding: {len(vector)} dimensions")
print(f"First 5 values: {vector[:5]}")

In [ ]:
# Local embeddings (Ollama)
from langchain_ollama import OllamaEmbeddings

try:
    local_embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vector = local_embeddings.embed_query("machine learning")
    print(f"Ollama embedding: {len(vector)} dimensions")
    print(f"First 5 values: {vector[:5]}")
except Exception as e:
    print(f"Ollama embeddings not available: {e}")
    print("Make sure Ollama is running and you've pulled nomic-embed-text:")
    print("  ollama pull nomic-embed-text")

---

## 5. Documents — LangChain's Data Structure

LangChain uses **Document** objects to hold text and metadata:

```mermaid
flowchart LR
    D["Document"] --> P["page_content: The text"]
    D --> M["metadata: Dict of attributes"]
```

| Field | Type | Description |
|---|---|---|
| `page_content` | `str` | The actual text content |
| `metadata` | `dict` | Key-value pairs (source, topic, etc.) |

In [ ]:
# Create a Document
doc = Document(
    page_content="Random Forest is an ensemble learning method that builds multiple decision trees.",
    metadata={
        "topic": "Random Forest",
        "category": "ensemble methods",
        "difficulty": "intermediate"
    }
)

print("Content:", doc.page_content)
print("Metadata:", doc.metadata)

---

## 6. Building a Data Science Knowledge Base

Let's create a knowledge base of Data Science concepts:

In [ ]:
# Data Science Knowledge Base
knowledge_base = [
    Document(
        page_content="Linear Regression models the relationship between a dependent variable and one or more independent variables by fitting a straight line. It assumes a linear relationship and minimizes the sum of squared residuals. Commonly used for predicting continuous values like house prices or sales forecasts.",
        metadata={"topic": "Linear Regression", "category": "regression", "difficulty": "beginner"}
    ),
    Document(
        page_content="Logistic Regression is a classification algorithm that predicts the probability of a binary outcome. Despite its name, it's used for classification, not regression. It uses the sigmoid function to map predictions to probabilities between 0 and 1. Commonly used for spam detection and medical diagnosis.",
        metadata={"topic": "Logistic Regression", "category": "classification", "difficulty": "beginner"}
    ),
    Document(
        page_content="Decision Trees are tree-shaped models that make predictions by asking a series of yes/no questions about features. They split data recursively based on feature values. Easy to interpret but prone to overfitting. Used in credit scoring, medical diagnosis, and customer segmentation.",
        metadata={"topic": "Decision Trees", "category": "tree-based", "difficulty": "beginner"}
    ),
    Document(
        page_content="Random Forest is an ensemble method that builds multiple decision trees and combines their predictions. Each tree is trained on a random subset of data and features. Reduces overfitting compared to single decision trees. Used for feature importance, classification, and regression tasks.",
        metadata={"topic": "Random Forest", "category": "ensemble methods", "difficulty": "intermediate"}
    ),
    Document(
        page_content="K-Means Clustering partitions data into K groups where each point belongs to the cluster with the nearest centroid. It iteratively assigns points to clusters and updates centroids. Fast and scalable but requires specifying K in advance. Used for customer segmentation and image compression.",
        metadata={"topic": "K-Means", "category": "clustering", "difficulty": "intermediate"}
    ),
    Document(
        page_content="Principal Component Analysis (PCA) reduces the dimensionality of data by finding the directions of maximum variance. It transforms high-dimensional data into fewer dimensions while preserving as much information as possible. Used for data visualization, noise reduction, and feature extraction.",
        metadata={"topic": "PCA", "category": "dimensionality reduction", "difficulty": "intermediate"}
    ),
    Document(
        page_content="Cross-validation is a technique for evaluating model performance by splitting data into training and validation sets multiple times. K-fold cross-validation divides data into K folds, training on K-1 and testing on 1. Helps detect overfitting and provides more reliable performance estimates.",
        metadata={"topic": "Cross Validation", "category": "model evaluation", "difficulty": "intermediate"}
    ),
    Document(
        page_content="A Confusion Matrix is a table that shows the performance of a classification model by comparing predicted labels against actual labels. It contains four values: True Positives, True Negatives, False Positives, and False Negatives. Foundation for metrics like precision, recall, and F1-score.",
        metadata={"topic": "Confusion Matrix", "category": "model evaluation", "difficulty": "beginner"}
    ),
    Document(
        page_content="Precision measures the accuracy of positive predictions. It's the ratio of true positives to all predicted positives. High precision means fewer false positives. Important when the cost of false positives is high, like in spam detection where you don't want to mark legitimate emails as spam.",
        metadata={"topic": "Precision", "category": "model evaluation", "difficulty": "beginner"}
    ),
    Document(
        page_content="Recall (Sensitivity) measures the ability to find all positive instances. It's the ratio of true positives to all actual positives. High recall means fewer false negatives. Important when missing positive cases is costly, like in disease detection where you don't want to miss sick patients.",
        metadata={"topic": "Recall", "category": "model evaluation", "difficulty": "beginner"}
    ),
    Document(
        page_content="F1 Score is the harmonic mean of precision and recall. It balances both metrics into a single number ranging from 0 to 1. Useful when you need to balance precision and recall, especially with imbalanced datasets. F1 = 2 * (precision * recall) / (precision + recall).",
        metadata={"topic": "F1 Score", "category": "model evaluation", "difficulty": "beginner"}
    ),
]

print(f"Knowledge base created with {len(knowledge_base)} documents")
print(f"\nTopics:")
for doc in knowledge_base:
    print(f"  - {doc.metadata['topic']} ({doc.metadata['category']})")

---

## 7. Document Splitting

For longer documents, you need to **split** them into smaller chunks before embedding.

### Why Split?

- Embedding models have **context limits**
- Smaller chunks = more **precise retrieval**
- Overlapping chunks preserve **context**

### RecursiveCharacterTextSplitter

The recommended splitter. It tries to split on paragraphs, then sentences, then words.

```mermaid
flowchart LR
    L["Long Document"] --> S["Text Splitter"]
    S --> C1["Chunk 1"]
    S --> C2["Chunk 2"]
    S --> C3["Chunk 3"]
```

In [ ]:
# Create a text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Maximum characters per chunk
    chunk_overlap=50,    # Overlap between chunks (preserves context)
    separators=["\n\n", "\n", ". ", " ", ""]  # Split priority
)

# Example: split a long document
long_text = """
Machine learning is a subset of artificial intelligence that enables systems to learn from data.
There are three main types: supervised learning, unsupervised learning, and reinforcement learning.

Supervised learning uses labeled data to train models. Common algorithms include linear regression,
decision trees, and neural networks. The model learns to map inputs to known outputs.

Unsupervised learning finds patterns in unlabeled data. K-means clustering and PCA are popular
examples. The model discovers hidden structures without explicit guidance.

Reinforcement learning trains agents through trial and error. The agent receives rewards or
penalties and learns optimal strategies. Used in game playing and robotics.
"""

chunks = splitter.split_text(long_text)
print(f"Original length: {len(long_text)} chars")
print(f"Split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  {chunk[:100]}...")

### 🔍 What Happened?

The splitter:
1. Tried to split on `\n\n` (paragraph breaks) first
2. If chunks were still too long, tried `\n` (line breaks)
3. Then tried `. ` (sentences), then spaces
4. Added **overlap** so context isn't lost at boundaries

> 💡 **For our knowledge base:** Our documents are short, so we don't need splitting. But for real-world documents (PDFs, web pages), splitting is essential.

---

## 8. Vector Stores

A **vector store** is a database optimized for storing and searching embeddings.

### What It Does

```mermaid
flowchart TD
    D["Documents"] --> E["Embedding Model"]
    E --> VS["Vector Store\n(ChromaDB)"]
    Q["Query"] --> QE["Embedding Model"]
    QE --> VS
    VS --> R["Similar Documents"]
```

### Why ChromaDB?

| Feature | ChromaDB |
|---|
| **Lightweight** | Runs in-memory, no server needed |
| **Local** | All data stays on your machine |
| **Free** | Open-source, no costs |
| **Easy** | Simple API, great for learning |
| **Persistent** | Can save to disk for later use |

In [ ]:
# Create a vector store from our knowledge base
# Using OpenAI embeddings
vectorstore = Chroma.from_documents(
    documents=knowledge_base,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="ds_knowledge_base"
)

print(f"Vector store created with {vectorstore._collection.count()} documents")

---

## 9. Similarity Search

Now the exciting part — searching by meaning!

### Top-K Retrieval

When you search, the vector store:
1. Embeds your query
2. Compares it to all stored vectors
3. Returns the **K most similar** documents

```mermaid
flowchart LR
    Q["Query"] --> VS["Vector Store"]
    VS --> T1["Result 1 (most similar)"]
    VS --> T2["Result 2"]
    VS --> T3["Result 3"]
```

In [ ]:
# Basic similarity search
query = "How do I evaluate a classification model?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: '{query}'")
print(f"\nTop 3 results:")
print("=" * 60)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}: {doc.metadata['topic']}")
    print(f"Category: {doc.metadata['category']}")
    print(f"Content: {doc.page_content[:120]}...")

In [ ]:
# Search with different queries
queries = [
    "grouping similar data points",
    "measuring prediction accuracy",
    "reducing number of features",
]

for query in queries:
    results = vectorstore.similarity_search(query, k=2)
    print(f"\nQuery: '{query}'")
    for doc in results:
        print(f"  -> {doc.metadata['topic']}")

### 🔍 What Happened?

Notice how semantic search finds the right topics even when the query uses **different words**:

| Query | Found | Why |
|---|---|---|
| "grouping similar data points" | K-Means | "grouping" ≈ "clustering" |
| "measuring prediction accuracy" | Confusion Matrix, F1 | "accuracy" ≈ "evaluation" |
| "reducing number of features" | PCA | "reducing features" ≈ "dimensionality reduction" |

Keyword search would have failed on these!

---

## 10. Similarity Search with Scores

You can also get the **similarity score** for each result:

| Score | Meaning |
|---|---|
| Close to 0 | Very similar |
| Close to 1 | Less similar |

In [ ]:
# Similarity search with scores (lower = more similar)
query = "What is precision and when is it important?"
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'")
print(f"\nResults with similarity scores:")
print("=" * 60)

for doc, score in results_with_scores:
    print(f"\n  Score: {score:.4f} | Topic: {doc.metadata['topic']}")
    print(f"  Content: {doc.page_content[:100]}...")

---

## 11. Metadata Filtering

You can filter search results by metadata attributes:

```mermaid
flowchart LR
    Q["Query + Filter"] --> VS["Vector Store"]
    VS --> F["Filtered Results"]
```

In [ ]:
# Search with metadata filter
query = "evaluation metrics"

# Filter: only search within 'model evaluation' category
filtered_results = vectorstore.similarity_search(
    query,
    k=3,
    filter={"category": "model evaluation"}
)

print(f"Query: '{query}' (filtered to 'model evaluation' category)")
print("\nResults:")
for doc in filtered_results:
    print(f"  - {doc.metadata['topic']}")

In [ ]:
# Filter by difficulty level
query = "classification algorithm"

# Only beginner-level topics
beginner_results = vectorstore.similarity_search(
    query,
    k=3,
    filter={"difficulty": "beginner"}
)

print(f"Query: '{query}' (filtered to 'beginner' difficulty)")
print("\nResults:")
for doc in beginner_results:
    print(f"  - {doc.metadata['topic']} ({doc.metadata['category']})")

---

## 12. Local Embeddings with Ollama

For fully local setup (no API key needed):

In [ ]:
# Create a local vector store with Ollama embeddings
try:
    local_embeddings = OllamaEmbeddings(model="nomic-embed-text")
    
    local_vectorstore = Chroma.from_documents(
        documents=knowledge_base,
        embedding=local_embeddings,
        collection_name="ds_knowledge_base_local"
    )
    
    # Search
    results = local_vectorstore.similarity_search("ensemble methods", k=2)
    print("Local Ollama search results:")
    for doc in results:
        print(f"  - {doc.metadata['topic']}")
        
except Exception as e:
    print(f"Ollama embeddings not available: {e}")
    print("\nTo use local embeddings:")
    print("  1. Install Ollama: https://ollama.com/download")
    print("  2. Pull embedding model: ollama pull nomic-embed-text")

---

## 13. Complete Semantic Search Pipeline

Let's build a reusable semantic search function:

In [ ]:
class DSSemanticSearch:
    """A semantic search engine for data science concepts."""
    
    def __init__(self, documents, embeddings_model=None):
        """Initialize with documents and optional custom embeddings."""
        if embeddings_model is None:
            embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
        
        self.embeddings = embeddings_model
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            collection_name="ds_search"
        )
    
    def search(self, query, k=3, category=None):
        """Search for similar concepts."""
        filter_dict = {"category": category} if category else None
        return self.vectorstore.similarity_search_with_score(
            query, k=k, filter=filter_dict
        )
    
    def display_results(self, query, k=3, category=None):
        """Search and display formatted results."""
        results = self.search(query, k, category)
        print(f"Query: '{query}'")
        if category:
            print(f"Filter: category='{category}'")
        print("=" * 60)
        for i, (doc, score) in enumerate(results):
            print(f"\n{i+1}. {doc.metadata['topic']} (score: {score:.4f})")
            print(f"   Category: {doc.metadata['category']}")
            print(f"   {doc.page_content[:100]}...")


# Create the search engine
search_engine = DSSemanticSearch(knowledge_base)

# Test it
search_engine.display_results("How to reduce data dimensions?")

In [ ]:
# Test with different queries
search_engine.display_results("predicting customer churn", k=2)
print("\n")
search_engine.display_results("measuring model performance", k=2, category="model evaluation")

---

## 14. API vs Local: Privacy, Cost, and Hardware

| | **OpenAI API** | **Local (Ollama)** |
|---|---|---|
| **Setup** | API key only | Install Ollama + pull model |
| **Cost** | Pay per 1M tokens (~$0.02) | Free |
| **Privacy** | Data sent to OpenAI servers | All data stays on your machine |
| **Quality** | High (1536 dims) | Good (384-768 dims) |
| **Speed** | ~50ms per embedding | 100-500ms (hardware dependent) |
| **Internet** | Required | Not required after model download |
| **Best for** | Production, accuracy-critical | Learning, privacy-sensitive, offline |

> 💡 **For this course:** Use OpenAI for demonstrations, Ollama for practice at home.

---

## ⚠️ Common Mistakes

| Mistake | Problem | Fix |
|---|---|---|
| **Embedding too long texts** | Exceeds model context limit | Split long documents first |
| **Not normalizing vectors** | Incorrect similarity scores | Use `embed_query()` which normalizes |
| **Embedding chat queries like chat messages** | Wrong model for the task | Use embedding model, not chat model |
| **Large chunk sizes** | Imprecise retrieval | Use 200-500 chars for most use cases |
| **No metadata** | Can't filter results | Always add topic/category metadata |
| **Forgetting overlap** | Missing context at boundaries | Set `chunk_overlap` to 10-20% of chunk_size |
| **Using cosine similarity on non-normalized vectors** | Wrong similarity scores | LangChain embeddings are pre-normalized |

---

## 🏋️ Exercises

Complete these exercises to solidify your understanding.

### Exercise 1: Expand the Knowledge Base

Add 3 more Data Science concepts to the knowledge base:
- XGBoost
- Neural Networks
- Gradient Descent

Then search for queries that should match these new concepts.

In [ ]:
# Exercise 1: Your code here!
#
# Steps:
# 1. Create new Document objects for XGBoost, Neural Networks, Gradient Descent
# 2. Add them to a new vector store
# 3. Search for: "boosting algorithm", "deep learning", "optimization"
# 4. Verify the results make sense


### Exercise 2: Multi-Category Search Engine

Build a search function that:
1. Takes a query
2. Searches across **all categories** in parallel
3. Returns the top result from each category
4. Ranks them by relevance

This simulates searching across different parts of a knowledge base.

In [ ]:
# Exercise 2: Your code here!
#
# Steps:
# 1. Get all unique categories from the knowledge base
# 2. For each category, filter and search
# 3. Collect top results from each
# 4. Sort by similarity score


### Exercise 3: Document Splitting Comparison

Compare different chunk sizes and overlaps:
1. Create a long text (at least 500 words about a data science topic)
2. Split it with chunk_size=100, 200, 500
3. Compare the number of chunks and average chunk length
4. Which chunk size gives the best retrieval results?

In [ ]:
# Exercise 3: Your code here!
#
# Steps:
# 1. Write or paste a long text (500+ words)
# 2. Create splitters with different chunk_sizes
# 3. Split and compare results
# 4. Create vector stores and test search quality


### 🌟 Challenge: Semantic Search vs Keyword Search

Build a comparison that demonstrates when semantic search beats keyword search:

1. Create 10 test queries designed to **trick** keyword search:
   - "handling empty values" (should find: missing data, NaN, imputation)
   - "combining multiple models" (should find: ensemble, bagging, boosting)
   - "visualizing high-dimensional data" (should find: PCA, t-SNE)

2. For each query, run both:
   - **Keyword search**: `doc.page_content.lower().contains(query_words)`
   - **Semantic search**: `vectorstore.similarity_search(query)`

3. Compare which method finds the right document
4. Create a results table showing success rates

In [ ]:
# 🌟 Challenge: Your code here!
#
# Suggested queries:
test_queries = [
    ("handling empty values", "Cross Validation"),
    ("combining multiple models", "Random Forest"),
    ("visualizing high-dimensional data", "PCA"),
    ("measuring positive predictions", "Precision"),
    ("finding groups in data", "K-Means"),
]
#
# For each query:
# 1. Check if expected topic appears in keyword search results
# 2. Check if expected topic appears in semantic search results
# 3. Compare success rates


---

## 📝 Key Takeaways

| Concept | What It Is | Key Insight |
|---|---|---|
| **Embeddings** | Text → vector (numbers) | Capture meaning, not just words |
| **Cosine Similarity** | Angle between vectors | 0 = identical, 1 = unrelated |
| **Embedding Model** | Converts text to vectors | Use `embed_query()` for search, `embed_documents()` for storage |
| **Document** | Text + metadata container | Always add metadata for filtering |
| **Text Splitter** | Breaks long text into chunks | Use `RecursiveCharacterTextSplitter` with overlap |
| **Vector Store** | Database for embeddings | ChromaDB for local learning, Pinecone/Weaviate for production |
| **Similarity Search** | Find semantically similar docs | Top-K retrieval with optional metadata filtering |

### The Complete Pipeline

```mermaid
flowchart TD
    D["Documents"] --> S["Text Splitter"]
    S --> E["Embedding Model"]
    E --> VS["Vector Store (ChromaDB)"]
    Q["User Query"] --> QE["Embedding Model"]
    QE --> VS
    VS --> R["Top-K Similar Documents"]
```

### When to Use What

| Scenario | Approach |
|---|---|
| Quick prototype | OpenAI embeddings + ChromaDB |
| Privacy-sensitive | Ollama embeddings + ChromaDB |
| Large-scale production | OpenAI embeddings + Pinecone/Weaviate |
| Offline/air-gapped | Ollama embeddings + ChromaDB |

---

## 🚀 Next Steps

| Notebook | Topic | What You'll Learn |
|---|---|---|
| **01** | Introduction | What is LangChain? |
| **02** | Models, Prompts & Messages | Prompt engineering + structured output |
| **03** | LCEL & Chains | Pipeline composition |
| **04** | Embeddings & Vector Stores | You are here! |
| **05** | RAG Applications | Question answering over documents |
| **06** | Tools & Agents | AI that can take actions |
| **07** | Advanced Project | Build a complete data science assistant |

---

🎉 **Excellent work!** You've mastered embeddings and vector stores.

Next up: **RAG Applications** — where you'll combine retrieval with generation to answer questions over your knowledge base! 🚀